# Experiment D: Multi-XGBoost Rank Ensemble

이 노트북은 가장 성능이 높았던 `pit-1.ipynb`의 물리적 특성 공학(Feature Engineering)을 베이스로 하며,
성격이 다른 3가지 **XGBoost 모델**을 앙상블하여 일반화 성능을 극대화합니다.

### 핵심 전략:
1. **Proven Features:** `pit-1.ipynb`에서 검증된 5대 물리 지표 활용
2. **XGBoost Diversification:** 
    - **XGB_Standard**: 균형 잡힌 베이스 모델
    - **XGB_Deep**: 복잡한 상호작용을 파악하는 깊은 모델
    - **XGB_Conservative**: 강한 규제로 오버피팅을 방지하는 안정적 모델
3. **Rank Averaging**: 각 모델의 확률 순위를 평균하여 AUC 최적화

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata
import gc
import warnings

warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/playground-series-s6e5/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train Shape: {train.shape}, Test Shape: {test.shape}')

## 1. Feature Engineering (From pit-1.ipynb)
성능이 검증된 물리 지표들을 생성합니다.

In [ ]:
def pit1_engineering(df):
    # 타이어 종류별 예상 수명
    compound_mean_life = {'HARD': 25, 'MEDIUM': 18, 'SOFT': 12, 'INTERMEDIATE': 20, 'WET': 15}
    df['Expected_Life'] = df['Compound'].map(compound_mean_life).fillna(20)
    
    # 1. 상대적 타이어 수명
    df['Relative_TyreLife'] = df['TyreLife'] / df['Expected_Life']
    # 2. 경기 막바지 여부 (85% 이상)
    df['Is_Final_Laps'] = (df['RaceProgress'] > 0.85).astype(int)
    # 3. 마모 모멘텀
    df['Degradation_Momentum'] = df['TyreLife'] * df['Cumulative_Degradation']
    # 4. 랩당 마모율
    df['Degradation_per_Lap'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
    # 5. 랩타임 변화량 절댓값
    df['Abs_LapTime_Delta'] = df['LapTime_Delta'].abs()
    
    df.drop(['Expected_Life'], axis=1, inplace=True)
    return df

train = pit1_engineering(train)
test = pit1_engineering(test)

# 범주형 변수 라벨 인코딩
cat_features = ['Driver', 'Compound', 'Race', 'Year']
le = LabelEncoder()
for col in cat_features:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

drop_cols = ['id', 'PitNextLap']
features = [c for c in train.columns if c not in drop_cols]
print("Feature Engineering Complete with pit-1.ipynb logic.")

## 2. Multi-XGBoost Model Definition
다양한 시각으로 데이터를 바라보는 3가지 성격의 XGBoost를 정의합니다.

In [ ]:
models_info = {
    'XGB_Standard': {
        'n_estimators': 1200, 'learning_rate': 0.05, 'max_depth': 6, 
        'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42
    },
    'XGB_Deep': {
        'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 9, 
        'subsample': 0.7, 'colsample_bytree': 0.7, 'random_state': 2024
    },
    'XGB_Conservative': {
        'n_estimators': 1500, 'learning_rate': 0.02, 'max_depth': 4, 
        'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'random_state': 777
    }
}

X = train[features]
y = train['PitNextLap']
groups = train['Race']
kf = GroupKFold(n_splits=5)

## 3. Training & Validation
각 모델에 대해 GroupKFold 교차 검증을 수행합니다.

In [ ]:
oof_dict = {}
test_dict = {}

for name, params in models_info.items():
    print(f"--- Training {name} ---")
    model = XGBClassifier(**params, eval_metric='auc', tree_method='hist')
    
    oof_preds = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y, groups)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test[features])[:, 1] / 5
        
    oof_dict[name] = oof_preds
    test_dict[name] = test_preds
    print(f"{name} OOF AUC: {roc_auc_score(y, oof_preds):.4f}")

## 4. Final Rank Ensemble
각 XGBoost의 예측 결과를 Rank Averaging으로 결합합니다.

In [ ]:
print("\n--- Calculating Final Ensemble ---")
final_oof = (
    rankdata(oof_dict['XGB_Standard']) +
    rankdata(oof_dict['XGB_Deep']) +
    rankdata(oof_dict['XGB_Conservative'])
) / 3

final_test = (
    rankdata(test_dict['XGB_Standard']) +
    rankdata(test_dict['XGB_Deep']) +
    rankdata(test_dict['XGB_Conservative'])
) / 3

print(f"Multi-XGB Ensemble OOF AUC: {roc_auc_score(y, final_oof):.4f}")

submission['PitNextLap'] = final_test
submission.to_csv('submission_xgb_ensemble.csv', index=False)
print("Submission file 'submission_xgb_ensemble.csv' saved.")